In [ ]:
import pandas as pd

import sys
sys.path.append('../')

In [ ]:
from constants import brussels_codes_muni_dict

# VALIDATION

In [ ]:
pop = pd.read_csv('output/workers_with_company_cars.csv')

In [ ]:
def extract_municipality_code(sector_id):
    """Extract 5-digit NIS municipality code from sector ID string."""
    return str(sector_id)[:5]

def parse_age_bracket(age_str):
    """Extract lower bound from age bracket string like '25-29', '100+'."""
    if pd.isna(age_str):
        return 0
    age_str = str(age_str).strip()
    if '+' in age_str:
        return int(age_str.replace('+', ''))
    return int(age_str.split('-')[0])

In [ ]:
print(f'Total workers: {len(pop)}')
print(f'Workers with cars: {pop["has_car"].sum()} ({pop["has_car"].mean()*100:.2f}%)')
print(f'Workers with company cars: {pop["has_company_car"].sum()} ({pop["has_company_car"].mean()*100:.2f}%)')
print(f'Percentage of car owners with company cars: {pop[pop["has_car"] == 1]["has_company_car"].mean()*100:.2f}%')

## Stats by municipality

In [ ]:
# Get the percentage of company car owners by municipality
pop['municipality'] = pop['assigned_sector'].apply(extract_municipality_code)
pop['municipality_name'] = pop['municipality'].astype(int).map(brussels_codes_muni_dict)
company_car_by_muni = pop.groupby('municipality_name')['has_company_car'].mean() * 100
print(company_car_by_muni.sort_values(ascending=False))

## Stats by age

In [ ]:
pop['age_bracket'] = pop['age'].apply(parse_age_bracket)
pop_with_car = pop[pop['has_car'] == 1]
age_stats = pop_with_car.groupby('age_bracket').agg(
    company_car_share=('has_company_car', 'mean'),
    sample_size=('has_company_car', 'size')
)
age_stats['company_car_share'] = age_stats['company_car_share'] * 100
print(age_stats.sort_values('company_car_share', ascending=False))

## Stats by economic sector

In [ ]:
company_car_by_industry = pop.groupby('industry')['has_company_car'].mean() * 100
print(company_car_by_industry.sort_values(ascending=False))